# Splitting strings into pieces.

**Python Foundation · An interactive experiment · 06**

Choose **Runtime → Run all** in Google Colab. Then change the text, separator, and `maxsplit` in the explorer. Press Enter after editing a text field to update it. Code cells are editable and run real Python.

`text.split(separator)` returns a list of strings. With no separator, Python splits on runs of whitespace. With an explicit separator, Python matches that exact substring.


In [ ]:
print("Comma:", "1,2,9".split(","))
print("Whitespace:", "1 2 9".split())
print("No matching whitespace:", "129".split())
print("Individual characters:", list("129"))
print("Individual integers:", list(map(int, "129")))


## Whitespace mode and exact-separator mode

| Expression | Result | Reason |
|---|---|---|
| `'  a   b  '.split()` | `['a', 'b']` | Runs of whitespace collapse; leading/trailing whitespace is ignored |
| `'  a   b  '.split(' ')` | `['', '', 'a', '', '', 'b', '', '']` | Each literal space is a separator; empty fields remain |
| `'a,,b,'.split(',')` | `['a', '', 'b', '']` | Consecutive/trailing separators create empty fields |
| `''.split()` | `[]` | No words in an empty string |
| `''.split(',')` | `['']` | One empty field with no comma |

`split()` also recognizes whitespace such as tabs and newlines. `split(' ')` matches only the literal space character. An empty explicit separator, `split('')`, raises `ValueError`. Separators can contain more than one character.


In [ ]:
text = "  a   b  "
print("Whitespace mode:", text.split())
print("Exact space:", text.split(" "))
print("Tabs and newlines:", "a\tb\nc".split())
print("Empty fields:", "a,,b,".split(","))
print("Multi-character separator:", "a::b::c".split("::"))
print("Empty text / whitespace:", "".split())
print("Empty text / comma:", "".split(","))
try:
    print("129".split(""))
except ValueError as error:
    print("Expected ValueError:", error)


## Limit the number of splits

`maxsplit` limits the number of splits, so there are at most `maxsplit + 1` pieces when `maxsplit` is nonnegative. A negative value means no limit. With an exact separator and `maxsplit=0`, the string is returned unsplit. In whitespace mode, `maxsplit=0` still removes leading whitespace, then retains the remainder (including trailing whitespace).


In [ ]:
print("All commas:", "a,b,c".split(","))
print("One comma:", "a,b,c".split(",", maxsplit=1))
print("Zero commas:", "a,b,c".split(",", maxsplit=0))
print("Whitespace / one:", "  a   b  ".split(maxsplit=1))
print("Whitespace / zero:", "  a   b  ".split(maxsplit=0))


## Explore split → pieces → join

Choose a preset or edit the text. In **Whitespace** mode the separator field is ignored. In **Exact separator** mode it is used literally (including a single space). Choose `-1` for unlimited splits.

The explorer uses `repr` to make spaces, tabs, newlines, and empty strings visible. Change the joining separator and see whether joining the pieces reproduces the original text. Join and split are not always exact inverses: whitespace mode discards spacing information.


In [ ]:
import ast
import html
import ipywidgets as widgets
from IPython.display import display

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

def panel(rows, note=""):
    escape = lambda value: html.escape(str(value))
    body = "".join(
        f'<tr><td style="padding:8px;vertical-align:top">{escape(label)}</td>'
        f'<td style="padding:8px;overflow-wrap:anywhere"><code>{escape(value)}</code></td></tr>'
        for label, value in rows
    )
    return (f'<div style="padding:20px;border:1px solid #cbd5e1;border-radius:14px;'
            f'background:#fff;color:#172b3a;font:15px/1.7 system-ui">'
            f'<table style="width:100%;text-align:left">{body}</table><p>{escape(note)}</p></div>')


def split_report(text, mode, separator, maxsplit, join_separator):
    effective_separator = None if mode == "Whitespace" else separator
    rows = [("Original string (repr)", repr(text)),
            ("Split separator", repr(effective_separator)), ("maxsplit", str(maxsplit))]
    try:
        pieces = text.split(effective_separator, maxsplit)
    except ValueError as error:
        rows.append(("ValueError", str(error)))
        return panel(rows, "Use a nonempty explicit separator, or choose Whitespace mode.")
    rows.append(("Result list", repr(pieces)))
    rows.append(("Number of pieces", str(len(pieces))))
    rows.extend((f"Piece {index}", repr(piece)) for index, piece in enumerate(pieces))
    joined = join_separator.join(pieces)
    rows.extend([("Join separator", repr(join_separator)),
                 ("Joined string", repr(joined)),
                 ("Joined == original", str(joined == text)),
                 ("Original after split", repr(text))])
    return panel(rows, "split returns a new list; join returns a new string. Whitespace splitting can discard the original spacing.")

presets = widgets.Dropdown(options=[("Digits / commas", "1,2,9"),
                                   ("Repeated spaces", "  a   b  "),
                                   ("Empty comma fields", "a,,b,"),
                                   ("Tabs and newlines", "a\tb\nc"),
                                   ("No separators", "129"),
                                   ("Empty string", "")], description="Example:")
text_input = widgets.Textarea(value="1,2,9", description="Text:", continuous_update=False, layout=widgets.Layout(width="95%"))
mode_input = widgets.Dropdown(options=["Whitespace", "Exact separator"], value="Exact separator", description="Mode:")
split_separator = widgets.Text(value=",", description="Split with:", continuous_update=False)
limit_input = widgets.IntSlider(value=-1, min=-1, max=6, description="maxsplit:")
join_separator = widgets.Text(value=",", description="Join with:", continuous_update=False)
split_view = widgets.HTML()
split_reset = widgets.Button(description="Reset")

def update_split(change=None):
    split_separator.disabled = mode_input.value == "Whitespace"
    split_view.value = split_report(text_input.value, mode_input.value, split_separator.value, limit_input.value, join_separator.value)

def select_split_example(change):
    text_input.value = change["new"]

def reset_split(button):
    presets.value = "1,2,9"
    text_input.value = "1,2,9"
    mode_input.value = "Exact separator"
    split_separator.value = ","
    limit_input.value = -1
    join_separator.value = ","
    update_split()

presets.observe(select_split_example, names="value")
for control in (text_input, mode_input, split_separator, limit_input, join_separator):
    control.observe(update_split, names="value")
split_reset.on_click(reset_split)
update_split()
display(widgets.VBox([presets, text_input, mode_input, split_separator, limit_input, join_separator, split_reset, split_view]))


## Characters or numbers?

For `'129'`, `split()` returns `['129']`, because there is no whitespace separator. Use `list(text)` for individual characters or `list(map(int, text))` for individual digit values. For comma-separated numbers such as `'12,30,9'`, split first and then convert each piece.

Try changing the examples below. Predict the list before running the cell.


In [ ]:
digit_text = "129"
number_text = "12,30,9"
print("split:", digit_text.split())
print("Characters:", list(digit_text))
print("Digit values:", list(map(int, digit_text)))
print("Number fields:", number_text.split(","))
print("Number values:", list(map(int, number_text.split(","))))

original = "  hello   world  "
normalized = " ".join(original.split())
print("Original:", repr(original))
print("Joined words:", repr(normalized))
print("Exactly the same?", normalized == original)


## Remember

- `split()` divides text on runs of whitespace.
- `split(separator)` uses the exact separator and can preserve empty fields.
- `list(text)` collects individual characters.
- `join` combines strings; `split` separates a string. Both return new objects.

Practice: [LeetCode 66 — Plus One](https://github.com/Lawson-Dong/leetcode-solution/tree/main/easy/066).

References: [Python str.split](https://docs.python.org/3/library/stdtypes.html#str.split), [Python str.join](https://docs.python.org/3/library/stdtypes.html#str.join).
